## Import Package and Setting

In [ ]:
import cv2
import numpy as np
import os
import shutil
import matplotlib.pyplot as plt
from PIL import Image

# SDK setting
SDK_dir = "C:\\Qualcomm\\AIStack\\QAIRT\\2.35.0.250530"       # Specify what's QNN SDK used
SDK_lib_dir = SDK_dir + "\\lib\\arm64x-windows-msvc"
SDK_v73_skel = SDK_dir + "\\lib\\hexagon-v73\\unsigned"
SDK_v81_skel = SDK_dir + "\\lib\\hexagon-v81\\unsigned"
SDK_bin_dir = SDK_dir + "\\bin\\aarch64-windows-msvc"

In [ ]:
!pip install -r ../requirements.txt

## Setup Artifacts

In [ ]:
%%cmd
mkdir RealEsrgan

In [ ]:
image_name = ".\\assets\\demo_512_512.jpg"
input_image = cv2.imread(image_name)
print(input_image.shape)
input_image_clip = np.clip(input_image, 0, 255) 
input_image_clip_norm = input_image_clip / 255.0
input_image_clip_norm.astype(np.float32).tofile("RealEsrgan\input_512_512.raw")

with open("RealEsrgan/input_list.txt","w") as file:
    for raw in os.listdir("RealEsrgan/"):
        if "input_512_512.raw" in raw:
            file.write(raw)


## Setup QNN env

In [ ]:
# Copy necessary libraries to a common location
libs = ["QnnCpu.dll","QnnHtp.dll", "QnnSystem.dll", "QnnHtpNetRunExtensions.dll", "QnnHtpPrepare.dll", "QnnHtpV73Stub.dll", "QnnHtpV81Stub.dll", "QnnModelDlc.dll"]
for lib in libs:
    shutil.copy(SDK_lib_dir + "\\" + lib, "RealEsrgan")

# Copy V73 Skel
skel_libs = ["libqnnhtpv73.cat", "libQnnHtpV73.so", "libQnnHtpV73Skel.so"]
for lib in skel_libs:
    shutil.copy(SDK_v73_skel + "\\" + lib, "RealEsrgan")
# Copy V81 Skel
skel_libs = ["libqnnhtpv81.cat", "libQnnHtpV81.so", "libQnnHtpV81Skel.so"]
for lib in skel_libs:
    shutil.copy(SDK_v81_skel + "\\" + lib, "RealEsrgan")

# Copy Bins
bins = ["qnn-net-run.exe", "qnn-profile-viewer.exe"]
for bin in bins:
    shutil.copy(SDK_bin_dir + "\\" + bin, "RealEsrgan")

In [ ]:
import json
# HTP backend extensions config file (htp_backend_extensions.json) example
htp_backend_extensions_data = {
    "backend_extensions": {
        "shared_library_path": "QnnHtpNetRunExtensions.dll",
        "config_file_path": "htp_config.json"
    }
}
 
# HTP backend config file (htp_config.json) example
htp_config_data = {
    "graphs": [
        {
            "vtcm_mb":8,
            "O":3.0,
            "graph_names":["realesrgan"],
        }
    ],
    "devices": [
        {
            "cores":[{
                "core_id": 0,
                "perf_profile": "burst",
                "rpc_control_latency":100
            }]
        }
    ]
}
 
# write the config files to a temporary location
with open(".\\RealEsrgan\\htp_backend_extensions.json", "w") as f:
    f.write(json.dumps(htp_backend_extensions_data, indent=2))
with open(".\\RealEsrgan\\htp_config.json", "w") as f:
    f.write(json.dumps(htp_config_data, indent=2))

### Model Generation
Please refer to AIHub [RealESRGAN page](https://github.com/quic/ai-hub-models/tree/main/qai_hub_models/models/real_esrgan_x4plus) and follow env. setting and installation
After installation please run below command to generate 2x input/output spec model on Compute device
```
python -m qai_hub_models.models.real_esrgan_x4plus.export --height 512 --width 512 --device "Snapdragon X Elite CRD" --target-runtime qnn_dlc --quantize w8a8 --num-calibration-samples 120 --scale-factor 2 --skip-inferencing --skip-profiling
```

## Convert & Inference Model on V73
if you are in v81 device please ignore this section

In [ ]:
%%cmd
cd RealEsrgan/
qnn-context-binary-generator --backend QnnHtp.dll --model QnnModelDlc.dll --dlc_path realesrgan.dlc --binary_file realesrgan_v73.serialized --config_file "htp_backend_extensions.json"

In [ ]:
%%cmd
cd RealEsrgan
qnn-net-run --retrieve_context ".\\output\\realesrgan_v73.serialized.bin" --input_list input_list.txt --backend QnnHtp.dll --output_dir output_dir  --config_file htp_backend_extensions.json --profiling_level basic --num_inferences 100

## Convert & Inference Model on v81
if you are in v73 device please ignore this section

In [ ]:
%%cmd
cd RealEsrgan/
qnn-context-binary-generator --backend QnnHtp.dll --model QnnModelDlc.dll --dlc_path realesrgan.dlc --binary_file realesrgan_v81.serialized --config_file "htp_backend_extensions.json"

In [ ]:
%%cmd
cd RealEsrgan
qnn-net-run --retrieve_context ".\\output\\realesrgan_v81.serialized.bin" --input_list input_list.txt --backend QnnHtp.dll --output_dir output_dir  --config_file htp_backend_extensions.json --profiling_level basic --num_inferences 100

## Post-process

In [ ]:
def crop_image(img, xy, scale_factor):
    '''Crop the image around the tuple xy

    Inputs:
    -------
    img: Image opened with PIL.Image
    xy: tuple with relative (x,y) position of the center of the cropped image
        x and y shall be between 0 and 1
    scale_factor: the ratio between the original image's size and the cropped image's size
    '''
    center = (img.size[0] * xy[0], img.size[1] * xy[1])
    new_size = (img.size[0] / scale_factor, img.size[1] / scale_factor)
    left = max (0, (int) (center[0] - new_size[0] / 2))
    right = min (img.size[0], (int) (center[0] + new_size[0] / 2))
    upper = max (0, (int) (center[1] - new_size[1] / 2))
    lower = min (img.size[1], (int) (center[1] + new_size[1] / 2))
    cropped_img = img.crop((left, upper, right, lower))
    return cropped_img

In [ ]:
def postprocess():
    plt.figure(dpi=100)
    
    #subplot(r,c) provide the no. of rows and columns
    _, axarr = plt.subplots(nrows=2,ncols=2,figsize=(8,8)) 
    
    results = []
    x = np.fromfile(".\\Realesrgan\\output_dir\\Result_0\\upscaled_image.raw", dtype=np.float32).reshape((1024, 1024, 3))
    x = x * 255
    x = cv2.cvtColor(x, cv2.COLOR_RGB2BGR) 
    x = x.astype(int)
    x = np.clip(x,0,255).astype(np.uint8)
    results.append(x)

    inp = cv2.imread("assets\\demo_512_512.jpg")
    inp = cv2.resize(inp,(1024,1024))
    inp = cv2.cvtColor(inp, cv2.COLOR_RGB2BGR) 

    axarr[0,0].imshow(inp)
    axarr[0,1].imshow(results[0])
    axarr[1,0].imshow(crop_image(Image.fromarray(inp), (0.60, 0.50), 3.0))
    axarr[1,1].imshow(crop_image(Image.fromarray(results[0]), (0.60, 0.50), 3.0))
    
    
    axarr[0,0].title.set_text('Input image')
    axarr[0,1].title.set_text('HTP')
    axarr[1,0].title.set_text('zoom input')
    axarr[1,1].title.set_text('zoom HTP')
postprocess()

# Profile

In [ ]:
%%cmd
cd RealEsrgan
qnn-profile-viewer --input_log output_dir/qnn-profiling-data.log
